In [1]:
# Cell 1 — imports & config
import numpy as np
import pandas as pd
import s3fs

SYMBOL = "BTCUSDT"
SCORED_ROOT = "s3://tradebot-config-tokyo/data/stageB/dataset=v1/scored"
SPLITS = ["train", "val", "test"]

fs = s3fs.S3FileSystem()

In [2]:
# Cell 2 — list scored paths
def list_scored_paths(split: str):
    base = f"{SCORED_ROOT.rstrip('/')}/symbol={SYMBOL}/split={split}/parquet"
    pattern = base.replace("s3://", "") + "/*.parquet"
    paths = sorted([f"s3://{p}" for p in fs.glob(pattern)])
    return base, paths

scored = {}
for sp in SPLITS:
    base, paths = list_scored_paths(sp)
    print(sp, "base:", base)
    print("  exists?", fs.exists(base))
    print("  n_paths:", len(paths))
    if paths:
        print("  first:", paths[0].split("/")[-1], " last:", paths[-1].split("/")[-1])
    scored[sp] = {"base": base, "paths": paths}

train base: s3://tradebot-config-tokyo/data/stageB/dataset=v1/scored/symbol=BTCUSDT/split=train/parquet
  exists? True
  n_paths: 90
  first: part-000000.parquet  last: part-000089.parquet
val base: s3://tradebot-config-tokyo/data/stageB/dataset=v1/scored/symbol=BTCUSDT/split=val/parquet
  exists? True
  n_paths: 10
  first: part-000000.parquet  last: part-000009.parquet
test base: s3://tradebot-config-tokyo/data/stageB/dataset=v1/scored/symbol=BTCUSDT/split=test/parquet
  exists? True
  n_paths: 10
  first: part-000000.parquet  last: part-000009.parquet


In [3]:
# Cell 3 — quick schema check on first file of each split
COLS_NEEDED = ["id_t", "stgB_s", "stgB_allow", "stgB_thr_s"]  # add others if you want

def read_head(path, cols):
    d = pd.read_parquet(path, engine="pyarrow", columns=[c for c in cols if c is not None])
    return d.head(3), d.dtypes

for sp in SPLITS:
    paths = scored[sp]["paths"]
    if not paths:
        continue
    dhead, dtypes = read_head(paths[0], COLS_NEEDED)
    print("\n==", sp, "==")
    print("cols:", list(dhead.columns))
    print("dtypes:\n", dtypes)
    display(dhead)


== train ==
cols: ['id_t', 'stgB_s', 'stgB_allow', 'stgB_thr_s']
dtypes:
 id_t          datetime64[ns, UTC]
stgB_s                    float32
stgB_allow                   int8
stgB_thr_s                float32
dtype: object


,id_t,stgB_s,stgB_allow,stgB_thr_s
0,2024-01-01 00:00:00+00:00,0.106717,0,0.397767
1,2024-01-01 00:00:30+00:00,0.243745,0,0.397767
2,2024-01-01 00:01:00+00:00,0.136461,0,0.397767



== val ==
cols: ['id_t', 'stgB_s', 'stgB_allow', 'stgB_thr_s']
dtypes:
 id_t          datetime64[ns, UTC]
stgB_s                    float32
stgB_allow                   int8
stgB_thr_s                float32
dtype: object


,id_t,stgB_s,stgB_allow,stgB_thr_s
0,2025-07-02 00:00:00+00:00,-0.068287,0,0.397767
1,2025-07-02 00:00:30+00:00,-0.085631,0,0.397767
2,2025-07-02 00:01:00+00:00,-0.099127,0,0.397767



== test ==
cols: ['id_t', 'stgB_s', 'stgB_allow', 'stgB_thr_s']
dtypes:
 id_t          datetime64[ns, UTC]
stgB_s                    float32
stgB_allow                   int8
stgB_thr_s                float32
dtype: object


,id_t,stgB_s,stgB_allow,stgB_thr_s
0,2025-09-01 00:00:00+00:00,-0.010141,0,0.397767
1,2025-09-01 00:00:30+00:00,-0.304638,0,0.397767
2,2025-09-01 00:01:00+00:00,-0.070865,0,0.397767


In [4]:
# Cell 4 — core scan: volume, allow_rate, stats s/thr, coherence allow vs rule, time span
def scan_split(paths, cols=("id_t","stgB_s","stgB_allow","stgB_thr_s"), max_files=None):
    n_rows = 0
    n_allow = 0

    # for quantiles approx (reservoir)
    RES_K = 200_000
    res_s = np.empty(0, dtype=np.float32)
    rng = np.random.default_rng(123)

    # spans
    t_min = None
    t_max = None
    day_set = set()

    # coherence checks
    n_bad_allow = 0
    n_bad_nan = 0

    # threshold seen
    thr_values = set()

    it = paths if max_files is None else paths[:max_files]
    for p in it:
        d = pd.read_parquet(p, engine="pyarrow", columns=list(cols))

        # normalize types
        ts = pd.to_datetime(d["id_t"], utc=True, errors="coerce")
        s = pd.to_numeric(d["stgB_s"], errors="coerce")
        allow = pd.to_numeric(d["stgB_allow"], errors="coerce").fillna(0).astype(np.int8)
        thr = pd.to_numeric(d["stgB_thr_s"], errors="coerce")

        # stats
        n = len(d)
        n_rows += n
        n_allow += int((allow == 1).sum())

        # track thr
        thr_values.update(set(np.unique(thr.dropna().astype(float).round(12).values)))

        # time span
        if ts.notna().any():
            mn = ts.min()
            mx = ts.max()
            t_min = mn if (t_min is None or mn < t_min) else t_min
            t_max = mx if (t_max is None or mx > t_max) else t_max
            # rough day coverage
            day_set.update(set(ts.dropna().dt.strftime("%Y-%m-%d").unique().tolist()))

        # coherence: allow should match (s >= thr) if rule is exactly that
        # tolerate NaN in s/thr
        mask_ok = s.notna() & thr.notna()
        rule = (s[mask_ok].to_numpy() >= thr[mask_ok].to_numpy())
        allow_m = (allow[mask_ok].to_numpy() == 1)
        n_bad_allow += int((rule != allow_m).sum())

        # NaN sanity
        n_bad_nan += int((ts.isna() | s.isna()).sum())

        # reservoir sample of s for quantiles
        s_arr = s.dropna().to_numpy(np.float32, copy=False)
        if s_arr.size:
            if res_s.size < RES_K:
                take = min(RES_K - res_s.size, s_arr.size)
                res_s = np.concatenate([res_s, s_arr[:take]])
            else:
                # random replace
                idx = rng.integers(0, n_rows, size=min(50_000, s_arr.size))
                rep = s_arr[:idx.size]
                res_s[idx % RES_K] = rep

    # quantiles
    qs = {}
    if res_s.size:
        qs = {q: float(np.quantile(res_s, q)) for q in [0.01, 0.05, 0.5, 0.95, 0.99]}

    out = {
        "paths": len(it),
        "rows": n_rows,
        "allow_rows": n_allow,
        "allow_rate": (n_allow / n_rows) if n_rows else np.nan,
        "t_min": t_min,
        "t_max": t_max,
        "n_unique_days": len(day_set),
        "thr_unique": sorted(list(thr_values))[:10] + (["..."] if len(thr_values) > 10 else []),
        "n_bad_allow_vs_rule": n_bad_allow,
        "n_bad_nan_basic": n_bad_nan,
        "s_quantiles": qs,
    }
    return out

rows = []
for sp in SPLITS:
    paths = scored[sp]["paths"]
    if not paths:
        continue
    rep = scan_split(paths)
    rows.append({"split": sp, **rep})

report = pd.DataFrame(rows)
display(report[["split","paths","rows","allow_rows","allow_rate","t_min","t_max","n_unique_days","thr_unique","n_bad_allow_vs_rule"]])

,split,paths,rows,allow_rows,allow_rate,t_min,t_max,n_unique_days,thr_unique,n_bad_allow_vs_rule
0,train,90,1537919,61751,0.040152,2024-01-01 00:00:00+00:00,2025-06-30 23:59:30+00:00,534,[0.397767275572],0
1,val,10,172800,519,0.003003,2025-07-02 00:00:00+00:00,2025-08-31 23:59:30+00:00,60,[0.397767275572],0
2,test,10,169920,1507,0.008869,2025-09-01 00:00:00+00:00,2025-10-30 23:59:30+00:00,59,[0.397767275572],0


In [5]:
# Cell 5 — deeper: allow_rate & s distribution by month (cheap scan)
def month_stats(paths, cols=("id_t","stgB_allow","stgB_s"), max_files=None):
    it = paths if max_files is None else paths[:max_files]
    agg = {}  # ym -> dict
    for p in it:
        d = pd.read_parquet(p, engine="pyarrow", columns=list(cols))
        ts = pd.to_datetime(d["id_t"], utc=True, errors="coerce")
        ym = ts.dt.strftime("%Y-%m")
        allow = pd.to_numeric(d["stgB_allow"], errors="coerce").fillna(0).astype(np.int8)
        s = pd.to_numeric(d["stgB_s"], errors="coerce")

        for key in ym.dropna().unique():
            m = (ym == key)
            n = int(m.sum())
            if n == 0:
                continue
            a = int((allow[m] == 1).sum())
            s_m = s[m].dropna()
            if key not in agg:
                agg[key] = {"rows":0,"allow":0,"s_sum":0.0,"s_n":0}
            agg[key]["rows"] += n
            agg[key]["allow"] += a
            if len(s_m):
                agg[key]["s_sum"] += float(s_m.sum())
                agg[key]["s_n"] += int(len(s_m))
    out = []
    for k in sorted(agg.keys()):
        v = agg[k]
        out.append({
            "month": k,
            "rows": v["rows"],
            "allow": v["allow"],
            "allow_rate": v["allow"]/v["rows"] if v["rows"] else np.nan,
            "s_mean": v["s_sum"]/v["s_n"] if v["s_n"] else np.nan,
        })
    return pd.DataFrame(out)

for sp in SPLITS:
    paths = scored[sp]["paths"]
    if not paths:
        continue
    ms = month_stats(paths)
    print("\n==", sp, "month stats ==")
    display(ms.tail(24))


== train month stats ==


,month,rows,allow,allow_rate,s_mean
0,2024-01,89280,4164,0.046640,-0.137000
1,2024-02,80640,2002,0.024826,-0.213260
2,2024-03,83519,7903,0.094625,0.029590
3,2024-04,83520,6154,0.073683,-0.001080
4,2024-05,86400,2055,0.023785,-0.140359
5,2024-06,83520,1001,0.011985,-0.245533
6,2024-07,89280,3279,0.036727,-0.086160
7,2024-08,86400,8339,0.096516,-0.013286
8,2024-09,83520,2298,0.027514,-0.149469
9,2024-10,89280,1639,0.018358,-0.194657



== val month stats ==


,month,rows,allow,allow_rate,s_mean
0,2025-07,86400,201,0.002326,-0.292208
1,2025-08,86400,318,0.003681,-0.294700



== test month stats ==


,month,rows,allow,allow_rate,s_mean
0,2025-09,86400,161,0.001863,-0.334518
1,2025-10,83520,1346,0.016116,-0.202548


In [6]:
# Cell 6 — threshold sensitivity curve: allow_rate vs thr (from stgB_s)
# NOTE: this is the key to decide if 0.398 is too strict for val/test.
THR_GRID = np.array([0.20,0.25,0.30,0.32,0.34,0.36,0.38,0.39,0.395,0.398,0.40,0.41,0.42,0.45,0.50], dtype=np.float64)

def allow_curve(paths, thr_grid=THR_GRID, cols=("stgB_s",), max_files=None):
    it = paths if max_files is None else paths[:max_files]
    counts = np.zeros_like(thr_grid, dtype=np.int64)
    total = 0
    for p in it:
        d = pd.read_parquet(p, engine="pyarrow", columns=list(cols))
        s = pd.to_numeric(d["stgB_s"], errors="coerce").to_numpy(np.float64, copy=False)
        s = s[np.isfinite(s)]
        total += s.size
        if s.size == 0:
            continue
        # vectorized: for each thr, count s>=thr
        # (thr_grid[None,:] compare) can be big, so do in blocks
        for i, thr in enumerate(thr_grid):
            counts[i] += int((s >= thr).sum())
    rates = counts / max(total, 1)
    return pd.DataFrame({"thr_s": thr_grid, "allow_rate_est": rates, "allow_count_est": counts, "total_seen": total})

curves = []
for sp in SPLITS:
    paths = scored[sp]["paths"]
    if not paths:
        continue
    c = allow_curve(paths)
    c["split"] = sp
    curves.append(c)

curves_df = pd.concat(curves, ignore_index=True)
display(curves_df.pivot(index="thr_s", columns="split", values="allow_rate_est"))

split,test,train,val
thr_s,,,
0.200,0.047428,0.165871,0.026215
0.250,0.033798,0.129154,0.016568
0.300,0.022705,0.094261,0.010017
0.320,0.019044,0.081526,0.007946
0.340,0.015966,0.069634,0.006383
0.360,0.013094,0.058612,0.004936
0.380,0.010676,0.048446,0.003877
0.390,0.009622,0.043678,0.003322
0.395,0.009163,0.041375,0.003102


In [7]:
# Cell 7 (optional) — compare logged stgB_allow vs recomputed allow(s>=thr_s) on a sample
# Useful if you ever suspect mismatch.
def allow_mismatch_sample(paths, n_rows=200_000):
    take = []
    got = 0
    for p in paths:
        d = pd.read_parquet(p, engine="pyarrow", columns=["stgB_s","stgB_thr_s","stgB_allow","id_t"])
        if len(d) == 0:
            continue
        if got + len(d) <= n_rows:
            take.append(d)
            got += len(d)
        else:
            take.append(d.iloc[: max(0, n_rows-got)])
            got = n_rows
            break
    x = pd.concat(take, ignore_index=True)
    s = pd.to_numeric(x["stgB_s"], errors="coerce")
    thr = pd.to_numeric(x["stgB_thr_s"], errors="coerce")
    allow = pd.to_numeric(x["stgB_allow"], errors="coerce").fillna(0).astype(np.int8)
    ok = s.notna() & thr.notna()
    rule = (s[ok].to_numpy() >= thr[ok].to_numpy())
    allow_m = (allow[ok].to_numpy() == 1)
    mismatch = (rule != allow_m).mean() if rule.size else np.nan
    return mismatch, x

for sp in SPLITS:
    paths = scored[sp]["paths"]
    if not paths:
        continue
    mm, _ = allow_mismatch_sample(paths, n_rows=200_000)
    print(sp, "mismatch_rate(rule vs stgB_allow) =", mm)

train mismatch_rate(rule vs stgB_allow) = 0.0
val mismatch_rate(rule vs stgB_allow) = 0.0
test mismatch_rate(rule vs stgB_allow) = 0.0
